In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
import missingno
from sklearn.preprocessing import OneHotEncoder , LabelEncoder ,OrdinalEncoder ,  StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.model_selection import train_test_split


import sys

import handling_script
import importlib

from handling_script import handling_data

importlib.reload(handling_script)

from sklearn.linear_model import LinearRegression 
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report , f1_score , recall_score , precision_score , roc_auc_score , roc_curve ,auc , confusion_matrix

from sklearn.metrics import mean_squared_error , root_mean_squared_error , r2_score 

from sklearn.model_selection import GridSearchCV, StratifiedKFold


In [ ]:
data = pd.read_csv(r"car_price.csv")

In [ ]:
df = data.copy()

handler= handling_data()

In [ ]:
df = handler.handle_data(df)
df = handler.handle_make(df)

In [ ]:
train ,test = train_test_split(df , shuffle=True , random_state = 42 , test_size = .2)

In [ ]:
handler.fit_handle_model(train)
train = handler.transform_handle_model(train)
test = handler.transform_handle_model(test)

handler.fit_handle_year(train)
train = handler.transform_handle_year(train)
test = handler.transform_handle_year(test)

handler.fit_handle_fueltype(train)
train = handler.transform_handle_fueltype(train)
test = handler.transform_handle_fueltype(test)

handler.fit_handle_enginesize(train)
train = handler.transform_handle_enginesize(train)
test = handler.transform_handle_enginesize(test)

handler.fit_handle_transmation(train)
train = handler.transform_handle_transmation(train)
test = handler.transform_handle_transmation(test)

handler.fit_handle_tax_mileage_mpg(train)
train = handler.transform_handle_tax_mileage_mpg(train)
test = handler.transform_handle_tax_mileage_mpg(test)

handler.fit_target_endode(train)
train = handler.transform_target_endode(train)
test = handler.transform_target_endode(test)

handler.fit_scale_encode_data(train)
train = handler.transform_scale_encode_data(train)
test = handler.transform_scale_encode_data(test)

handler.fit_handle_price(train)
train = handler.transform_handle_price(train)
test = handler.transform_handle_price(test)


In [ ]:
train.isna().sum()

In [ ]:
test.isna().sum()

In [ ]:
before_model = handler.before_model
before_model

In [ ]:
feature_named = before_model.named_steps['preproccesor'].get_feature_names_out()

cat_cols = [col for col in feature_named if 'cat_OH' in col or 'cat_ordinal' in col or 'remainder' in col]
cat_cols.remove('remainder__price')
cat_cols

* Detect and handle outliers — use boxplots or the IQR method

In [ ]:
handler.fit_detect_outliers(train.drop(columns=cat_cols))

In [ ]:
handler.view_outliers_with_values(train.drop(columns=cat_cols))

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(15,7))

axes = axes.flatten()

for i , col in enumerate(train.drop(columns=cat_cols)):
    sns.boxplot(data=train , x = col , ax = axes[i])

plt.tight_layout()
plt.show()

In [ ]:
cat_cols.append('num__tax')
cat_cols

In [ ]:
print(f'shape train before remove outlairs : {train.shape}')

In [ ]:
train = handler.remove_outliers_with_values(train , exclude_cols=cat_cols)
train

In [ ]:
print(f'shape train after remove outlairs : {train.shape}')

In [ ]:
print(f'shape test before remove outlairs : {test.shape}')

In [ ]:
test = handler.remove_outliers_with_values(test ,exclude_cols=cat_cols)
test

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(15,7))

axes = axes.flatten()

for i , col in enumerate(train.drop(columns=cat_cols)):
    sns.boxplot(data=train , x = col , ax = axes[i])

for j in range(i+1 , len(axes)):
    fig.delaxes(axes[j])
    
plt.tight_layout()
plt.show()

# Task 3 : Create Two Target Variables

In [ ]:

threshold_33 = train['remainder__price'].quantile(0.33)
threshold_66 = train['remainder__price'].quantile(0.66)

print("--- Price Category Thresholds ---")
print(f"Cheap: <= {threshold_33:.2f}")
print(f"Moderate: > {threshold_33:.2f} to <= {threshold_66:.2f}")
print(f"Expensive: > {threshold_66:.2f}")

price_sorted = train['remainder__price'].sort_values(ascending=True)

def categorize_price(x):
    if x <= threshold_33:
        return "Cheap"
    elif x <= threshold_66:
        return "Moderate"
    else:
        return "Expensive"

train['price_category'] = train['remainder__price'].apply(categorize_price)
test['price_category'] = test['remainder__price'].apply(categorize_price)

# 3. Show how many cars fall into each category (Requirement for the assignment)
print("--- Train Distribution ---")
print(train['price_category'].value_counts())

print("\n--- Test Distribution ---")
print(test['price_category'].value_counts())

In [ ]:
sns.histplot(train['remainder__price'], bins=50, kde=True)
plt.axvline(threshold_33, color='r')
plt.axvline(threshold_66, color='g')
plt.show()

# Task 4 : Model 1: Linear Regression

In [ ]:
xtrain = train.drop(columns=['remainder__price','price_category'])
ytrain = train['remainder__price']
xtest = test.drop(columns=['remainder__price','price_category'])
ytest = test['remainder__price']

In [ ]:
model_linear = LinearRegression()
model_linear.fit(xtrain, ytrain)
y_pred = model_linear.predict(xtest)


mse = mean_squared_error(ytest, y_pred)
rmse = root_mean_squared_error(ytest, y_pred)
r2 = r2_score(ytest, y_pred)

print(f'Mean Square Error : {mse}')
print(f'Root Mean Square Error : {rmse}')
print(f'R2 Score : {r2}')



In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8,6))

sns.scatterplot(x=ytest, y=y_pred)

plt.xlabel("Actual Price")
plt.ylabel("Predicted Price")
plt.title("Actual vs Predicted Prices")


max_val = max(ytest.max(), y_pred.max())
min_val = min(ytest.min(), y_pred.min())

plt.plot([min_val, max_val], [min_val, max_val], color='red')

plt.show()

In [ ]:
residuals = ytest - y_pred

plt.figure(figsize=(8,6))

sns.scatterplot(x=y_pred, y=residuals)

plt.axhline(y=0, color='red', linestyle='--')

plt.xlabel("Predicted Price")
plt.ylabel("Residuals (Actual - Predicted)")
plt.title("Residual Plot")

plt.show()

# Task 5 : Model 2: KNN Classification

In [ ]:
y_train_classification = train['price_category']
y_test_classification = test['price_category']

In [ ]:
param_grid = {
    'n_neighbors': [3, 5, 7, 9, 11, 13],
    'metric': ['euclidean', 'manhattan']
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

knn = KNeighborsClassifier()

grid = GridSearchCV(
    estimator=knn,
    param_grid=param_grid,
    cv=cv,
    scoring='accuracy',
    n_jobs=-1
)
grid.fit(xtrain, y_train_classification)

In [ ]:
results = pd.DataFrame(grid.cv_results_)

plt.figure(figsize=(8,5))

sns.lineplot(
    data=results,
    x='param_n_neighbors',
    y='mean_test_score',
    marker='o'
)

plt.title("Effect of n_neighbors on Accuracy")
plt.xlabel("n_neighbors")
plt.ylabel("CV Accuracy")
plt.show()

In [ ]:
plt.figure(figsize=(6,5))

sns.barplot(
    data=results,
    x='param_metric',
    y='mean_test_score'
)

plt.title("Metric Comparison (Euclidean vs Manhattan)")
plt.xlabel("Distance Metric")
plt.ylabel("CV Accuracy")
plt.show()

In [ ]:
pivot_table = results.pivot_table(
    values='mean_test_score',
    index='param_n_neighbors',
    columns='param_metric'
)
pivot_table

In [ ]:
plt.figure(figsize=(6,5))

sns.heatmap(pivot_table, annot=True, cmap="viridis")

plt.title("KNN Performance Heatmap")
plt.xlabel("Metric")
plt.ylabel("n_neighbors")
plt.show()

In [ ]:
best_model = grid.best_estimator_
print("Best Params:", grid.best_params_)

In [ ]:
ypred = best_model.predict(xtest)

In [ ]:
print("Accuracy:", accuracy_score(y_test_classification, ypred))
print("\nClassification Report:\n", classification_report(y_test_classification, ypred))

In [ ]:
cm = confusion_matrix(y_test_classification, ypred)
print("Confusion Matrix:\n", cm)

In [ ]:
sns.heatmap(cm , annot=True , fmt = '.2f' ,cmap='Blues')

* The Linear Regression model performed well in predicting continuous car prices using metrics such as RMSE and R² score.
The KNN Classification model performed well in categorizing cars into Cheap, Moderate, and Expensive classes using accuracy score.

* In regression, “better” means:

* Lower RMSE and MSE
* Higher R² score

* In classification, “better” means:

* Higher classification accuracy
* Better prediction of class labels

* Is classification easier than regression on this dataset? Why or why not?

* Yes, classification is generally easier in this dataset because predicting an exact car price is more difficult than predicting a price range category.

* Car prices are affected by many small variations and outliers, making regression more sensitive.
* However, classification simplifies the problem by grouping prices into categories, reducing prediction complexity.

* Does converting price into categories lose important information?

* Yes, converting prices into categories loses some information because exact numerical differences between cars disappear.

* For example, two cars priced at 15,000 and 19,000 may belong to the same category even though their prices are different.

* However, categorization simplifies the learning task and may improve classification performance.

In [ ]:
train.drop(columns = ['price_category']).corr()['remainder__price'].sort_values(ascending=False)

In [ ]:
xtrain_drop = xtrain.drop(columns=['num__engineSize'])
xtest_drop = xtest.drop(columns=['num__engineSize'])

In [ ]:
model_linear = LinearRegression()
model_linear.fit(xtrain_drop, ytrain)
y_pred = model_linear.predict(xtest_drop)


mse = mean_squared_error(ytest, y_pred)
rmse = root_mean_squared_error(ytest, y_pred)
r2 = r2_score(ytest, y_pred)

print(f'Mean Square Error : {mse}')
print(f'Root Mean Square Error : {rmse}')
print(f'R2 Score : {r2}')



* The performance did not drop significantly, which suggests that other features still contain similar information.

# Try running KNN without scaling — how much does performance change?

In [ ]:
scaler = before_model.named_steps['preproccesor'].named_transformers_['num']

num_cols = before_model.named_steps['preproccesor'].transformers_[0][2]
num_cols

In [ ]:
train_no_scale = train.copy()

train_no_scale[[f'num__{col}' for col in num_cols]] = scaler.inverse_transform(
    train[[f'num__{col}' for col in num_cols]]
)

test_no_scale = test.copy()

test_no_scale[[f'num__{col}' for col in num_cols]] = scaler.inverse_transform(
    test[[f'num__{col}' for col in num_cols]]
)

In [ ]:
xtrain_no_scale = train_no_scale.drop(columns=['remainder__price', 'price_category'])
xtest_no_scale = test_no_scale.drop(columns=['remainder__price', 'price_category'])

In [ ]:
knn_no_scale = KNeighborsClassifier(n_neighbors=9 , metric='manhattan')

knn_no_scale.fit(xtrain_no_scale, y_train_classification)

y_pred_no_scale = knn_no_scale.predict(xtest_no_scale)

acc_no_scale = accuracy_score(y_test_classification, y_pred_no_scale)


In [ ]:
print("Accuracy:", accuracy_score(y_test_classification, y_pred_no_scale))
print("\nClassification Report:\n", classification_report(y_test_classification, y_pred_no_scale))

In [ ]:
cm = confusion_matrix(y_test_classification, y_pred_no_scale)

print("Confusion Matrix:\n", cm)

In [ ]:
sns.heatmap(cm , annot=True , fmt = '.2f' ,cmap='Blues')

* KNN performance decreased noticeably without scaling because KNN depends on distance calculations.

* Features with larger numerical ranges, such as mileage, dominate the distance metric and negatively affect model performance.

# 3) Try a different threshold for your price categories — does KNN accuracy change significantly?

In [ ]:
threshold_25 = train['remainder__price'].quantile(0.25)
threshold_75 = train['remainder__price'].quantile(0.75)

print("--- Price Category Thresholds ---")
print(f"Cheap: <= {threshold_25:.2f}")
print(f"Moderate: > {threshold_25:.2f} to <= {threshold_75:.2f}")
print(f"Expensive: > {threshold_75:.2f}")

price_sorted = train['remainder__price'].sort_values(ascending=True)

def categorize_price(x):
    if x <= threshold_25:
        return "Cheap"
    elif x <= threshold_75:
        return "Moderate"
    else:
        return "Expensive"

train['price_category'] = train['remainder__price'].apply(categorize_price)
test['price_category'] = test['remainder__price'].apply(categorize_price)

# 3. Show how many cars fall into each category (Requirement for the assignment)
print("--- Train Distribution ---")
print(train['price_category'].value_counts())

print("\n--- Test Distribution ---")
print(test['price_category'].value_counts())


In [ ]:
sns.histplot(train['remainder__price'], bins=50, kde=True)
plt.axvline(threshold_25, color='r')
plt.axvline(threshold_75, color='g')
plt.show()

In [ ]:
xtrain = train.drop(columns=['remainder__price', 'price_category'])
xtest = test.drop(columns=['remainder__price', 'price_category'])

In [ ]:
y_train_classification = train['price_category']
y_test_classification = test['price_category']

In [ ]:
param_grid = {
    'n_neighbors': [3, 5, 7, 9, 11, 13],
    'metric': ['euclidean', 'manhattan']
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

knn = KNeighborsClassifier()

grid = GridSearchCV(
    estimator=knn,
    param_grid=param_grid,
    cv=cv,
    scoring='accuracy',
    n_jobs=-1
)
grid.fit(xtrain, y_train_classification)

In [ ]:
results = pd.DataFrame(grid.cv_results_)

plt.figure(figsize=(8,5))

sns.lineplot(
    data=results,
    x='param_n_neighbors',
    y='mean_test_score',
    marker='o'
)

plt.title("Effect of n_neighbors on Accuracy")
plt.xlabel("n_neighbors")
plt.ylabel("CV Accuracy")
plt.show()

In [ ]:
plt.figure(figsize=(6,5))

sns.barplot(
    data=results,
    x='param_metric',
    y='mean_test_score'
)

plt.title("Metric Comparison (Euclidean vs Manhattan)")
plt.xlabel("Distance Metric")
plt.ylabel("CV Accuracy")
plt.show()

pivot_table = results.pivot_table(
    values='mean_test_score',
    index='param_n_neighbors',
    columns='param_metric'
)

In [ ]:
plt.figure(figsize=(6,5))

sns.heatmap(pivot_table, annot=True, cmap="viridis")

plt.title("KNN Performance Heatmap")
plt.xlabel("Metric")
plt.ylabel("n_neighbors")
plt.show()

In [ ]:
best_model = grid.best_estimator_
best_model

In [ ]:
print("Best Params:", grid.best_params_)
ypred = best_model.predict(xtest)

In [ ]:
print("Accuracy:", accuracy_score(y_test_classification, ypred))
print("\nClassification Report:\n", classification_report(y_test_classification, ypred))

In [ ]:
cm = confusion_matrix(y_test_classification, ypred)
print("Confusion Matrix:\n", cm)

In [ ]:
sns.heatmap(cm , annot=True , fmt = '.2f' ,cmap='Blues')

* Changing the category thresholds slightly changed the KNN accuracy because the class distribution became different.

* More balanced classes usually improve classification performance, while highly imbalanced thresholds may reduce accuracy.